In [40]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)
import shap
import lime
import lime.lime_tabular


In [41]:
df = pd.read_csv("loan_dataset_100k.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nClass distribution:\n", df['loan_status'].value_counts())

Shape: (100000, 17)

Columns: ['applicant_income', 'coapplicant_income', 'loan_amount', 'loan_duration', 'credit_score', 'dti_ratio', 'savings_balance', 'interest_rate', 'age', 'employment_status', 'education_level', 'residence_type', 'loan_purpose', 'marital_status', 'credit_history', 'previous_default', 'loan_status']

Missing values:
 applicant_income      0
coapplicant_income    0
loan_amount           0
loan_duration         0
credit_score          0
dti_ratio             0
savings_balance       0
interest_rate         0
age                   0
employment_status     0
education_level       0
residence_type        0
loan_purpose          0
marital_status        0
credit_history        0
previous_default      0
loan_status           0
dtype: int64

Class distribution:
 loan_status
1    65001
0    34999
Name: count, dtype: int64


In [42]:
print("Numerical summary:")
df.describe().T

Numerical summary:


,count,mean,std,min,25%,50%,75%,max
applicant_income,100000.0,6.380142e+04,50529.648529,5000.0000,29618.250000,53855.0000,8.399375e+04,600000.00
coapplicant_income,100000.0,4.072103e+04,24127.589989,2801.0000,24218.750000,35005.0000,5.080625e+04,300000.00
loan_amount,100000.0,1.226845e+06,929499.648863,85433.0000,596272.750000,942444.5000,1.571098e+06,11354371.00
loan_duration,100000.0,5.331568e+01,17.184244,6.0000,42.000000,53.0000,6.500000e+01,126.00
credit_score,100000.0,6.498206e+02,79.789887,356.0000,593.000000,650.0000,7.070000e+02,900.00
dti_ratio,100000.0,3.253645e-01,0.226888,0.0281,0.165575,0.2507,4.096000e-01,1.00
savings_balance,100000.0,4.174382e+05,332852.471032,14754.0000,197529.500000,328932.5000,5.350438e+05,5098862.00
interest_rate,100000.0,1.100451e+01,1.990423,5.5000,9.620000,11.0000,1.238000e+01,18.69
age,100000.0,3.515310e+01,7.874988,21.0000,29.000000,34.0000,4.000000e+01,70.00
credit_history,100000.0,6.504700e-01,0.476824,0.0000,0.000000,1.0000,1.000000e+00,1.00


In [43]:
categorical_cols = [
    'employment_status',
    'education_level',
    'residence_type',
    'loan_purpose',
    'marital_status'
]

# We use a dictionary to store one encoder per column
# so we can reuse them in app.py at prediction time
encoders = {}

df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Save all encoders together as one file
joblib.dump(encoders, "encoder.pkl")
print("\n✅ Encoders saved to encoder.pkl")

employment_status: {'Employed': 0, 'Self-employed': 1, 'Student': 2, 'Unemployed': 3}
education_level: {'Graduate': 0, 'High School': 1, 'Postgraduate': 2}
residence_type: {'Mortgage': 0, 'Owned': 1, 'Rented': 2}
loan_purpose: {'Business': 0, 'Car': 1, 'Education': 2, 'Home': 3, 'Personal': 4}
marital_status: {'Divorced': 0, 'Married': 1, 'Single': 2}

✅ Encoders saved to encoder.pkl


In [44]:
X = df_encoded.drop(columns=['loan_status'])
y = df_encoded['loan_status']

# Save the exact feature order — app.py needs this
feature_names = X.columns.tolist()
joblib.dump(feature_names, "feature_names.pkl")
print("Features:", feature_names)
print("Target distribution:", y.value_counts().to_dict())

Features: ['applicant_income', 'coapplicant_income', 'loan_amount', 'loan_duration', 'credit_score', 'dti_ratio', 'savings_balance', 'interest_rate', 'age', 'employment_status', 'education_level', 'residence_type', 'loan_purpose', 'marital_status', 'credit_history', 'previous_default']
Target distribution: {1: 65001, 0: 34999}


In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keeps class balance in both splits
)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

Train size: (80000, 16)
Test size:  (20000, 16)


In [46]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler for use in app.py
joblib.dump(scaler, "scaler.pkl")
print("✅ Scaler saved to scaler.pkl")

✅ Scaler saved to scaler.pkl


In [47]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression ===")
print(f"Accuracy : {accuracy_score(y_test, lr_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, lr_probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lr_preds))

=== Logistic Regression ===
Accuracy : 0.9713
ROC-AUC  : 0.9971

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96      7000
           1       0.98      0.98      0.98     13000

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000



In [48]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1           # uses all CPU cores
)
rf_model.fit(X_train, y_train)   # RF doesn't need scaled data

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(f"Accuracy : {accuracy_score(y_test, rf_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, rf_probs):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rf_preds))

=== Random Forest ===
Accuracy : 0.9775
ROC-AUC  : 0.9982

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      7000
           1       0.98      0.98      0.98     13000

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000



In [49]:
lr_auc = roc_auc_score(y_test, lr_probs)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"Logistic Regression AUC : {lr_auc:.4f}")
print(f"Random Forest AUC       : {rf_auc:.4f}")

if rf_auc >= lr_auc:
    best_model      = rf_model
    best_model_name = "Random Forest"
    best_X_train    = X_train        # unscaled
    best_X_test     = X_test
else:
    best_model      = lr_model
    best_model_name = "Logistic Regression"
    best_X_train    = X_train_scaled
    best_X_test     = X_test_scaled

print(f"\n✅ Best model: {best_model_name}")

# Save best model and its name
joblib.dump(best_model, "best_model.pkl")
joblib.dump(best_model_name, "best_model_name.pkl")
joblib.dump(best_X_train, "X_train.pkl")   # LIME needs training data
print("✅ Model saved to best_model.pkl")

Logistic Regression AUC : 0.9971
Random Forest AUC       : 0.9982

✅ Best model: Random Forest
✅ Model saved to best_model.pkl


In [50]:
print("Building SHAP explainer (may take 30–60 seconds)...")

# TreeExplainer works for Random Forest — fast and accurate
# If best model is Logistic Regression, use LinearExplainer instead
if best_model_name == "Random Forest":
    shap_explainer = shap.TreeExplainer(best_model)
else:
    shap_explainer = shap.LinearExplainer(best_model, best_X_train)

# Quick sanity check on 5 rows
sample        = pd.DataFrame(best_X_test[:5], columns=feature_names)
shap_vals     = shap_explainer.shap_values(sample)

# For binary classification, TreeExplainer returns a list [class0, class1]
# We want class 1 (Approved)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

print("SHAP values shape:", shap_vals.shape)
print("Sample SHAP for first row:", dict(zip(feature_names, shap_vals[0].round(4))))

joblib.dump(shap_explainer, "shap_explainer.pkl")
print("✅ SHAP explainer saved to shap_explainer.pkl")

Building SHAP explainer (may take 30–60 seconds)...
SHAP values shape: (5, 16, 2)
Sample SHAP for first row: {'applicant_income': array([ 0.3065, -0.3065]), 'coapplicant_income': array([-0.0089,  0.0089]), 'loan_amount': array([-0.0133,  0.0133]), 'loan_duration': array([-0.0011,  0.0011]), 'credit_score': array([-0.1194,  0.1194]), 'dti_ratio': array([-0.1314,  0.1314]), 'savings_balance': array([-0.0156,  0.0156]), 'interest_rate': array([ 0.0191, -0.0191]), 'age': array([ 0.0004, -0.0004]), 'employment_status': array([-0.0601,  0.0601]), 'education_level': array([-0.0011,  0.0011]), 'residence_type': array([ 0.0002, -0.0002]), 'loan_purpose': array([-0.0049,  0.0049]), 'marital_status': array([-0.0007,  0.0007]), 'credit_history': array([ 0.2455, -0.2455]), 'previous_default': array([-0.0339,  0.0339])}
✅ SHAP explainer saved to shap_explainer.pkl


In [51]:
# LIME cannot be pickled — we save the training data instead
# and recreate the explainer in app.py at startup

import os

# X_train.pkl is already saved in Cell 10, just verify it
size = os.path.getsize("X_train.pkl")
print(f"✅ X_train.pkl exists — {size/1024:.1f} KB")
print(f"   Shape: {np.array(best_X_train).shape}")
print("\n⚠️  LIME explainer will be built at Flask startup instead of saved.")
print("   This is normal — LIME uses lambda functions that can't be pickled.")

✅ X_train.pkl exists — 9064.4 KB
   Shape: (80000, 16)

⚠️  LIME explainer will be built at Flask startup instead of saved.
   This is normal — LIME uses lambda functions that can't be pickled.


In [52]:
import os

expected_files = [
    "best_model.pkl",
    "best_model_name.pkl",
    "shap_explainer.pkl",
    "lime_explainer.pkl",
    "scaler.pkl",
    "encoder.pkl",
    "feature_names.pkl",
    "X_train.pkl"
]

print("Saved files check:")
for f in expected_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌ MISSING"
    size   = f"{os.path.getsize(f)/1024:.1f} KB" if exists else ""
    print(f"  {status}  {f}  {size}")

Saved files check:
  ✅  best_model.pkl  26399.2 KB
  ✅  best_model_name.pkl  0.0 KB
  ✅  shap_explainer.pkl  39394.0 KB
  ✅  lime_explainer.pkl  5.4 KB
  ✅  scaler.pkl  1.6 KB
  ✅  encoder.pkl  1.8 KB
  ✅  feature_names.pkl  0.3 KB
  ✅  X_train.pkl  9064.4 KB
